# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook freezes a simple, human-readable baseline for the **CTR / engagement opportunity** lane. It uses only the provided trailing-90-day starter slice. The output is a review queue, not an automatic edit instruction, and every conclusion below is observational.

## 1. Check two signals before writing the rule

**Signal A — CTR versus search position (FlyRank flag-linked).** If snippet opportunity is real, volume-qualified pages should generally earn lower CTR as their result position worsens. I require at least 500 impressions and exclude `avg_position == 0`, which means missing position data. **Verdict: MIXED.** The broad direction is present, especially after page one, but top-3, page-one, and striking-distance CTR are not perfectly monotonic in this cross-client slice. That warns me to compare pages only inside a coarse position bucket rather than assume a universal CTR curve.

**Signal B — impression volume (FlyRank quick-win signal).** If volume represents practical upside, higher-volume buckets should show more observed clicks and more pages with at least one click. **Verdict: CONFIRMED.** The table is descriptive, not causal; volume is used to prioritize an existing gap, never as proof that an edit will work.

All rate columns are percentage points: for example, `ctr = 0.76` means 0.76%.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
repo_root = Path.cwd().resolve()
while not (repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('Run this notebook from somewhere inside the repository.')
    repo_root = repo_root.parent

source_file = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(source_file)

assert df.shape == (30_000, 44)
assert df['content_id'].is_unique
assert (df['impressions_90d'] >= 1).all()

position_rows = df.loc[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()
position_rows['position_bucket'] = pd.cut(
    position_rows['avg_position'],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep'],
)
position_signal = (
    position_rows.groupby('position_bucket', observed=True)
    .agg(
        n=('content_id', 'size'),
        impressions=('impressions_90d', 'sum'),
        clicks=('clicks_90d', 'sum'),
        median_ctr_pct=('ctr', 'median'),
    )
    .reset_index()
)
position_signal['weighted_ctr_pct'] = (
    100 * position_signal['clicks'] / position_signal['impressions']
)
position_signal = position_signal[[
    'position_bucket', 'n', 'median_ctr_pct', 'weighted_ctr_pct'
]]

volume_rows = df.copy()
volume_rows['volume_bucket'] = pd.cut(
    volume_rows['impressions_90d'],
    bins=[0, 499, 2_999, 29_999, np.inf],
    labels=['1-499', '500-2,999', '3,000-29,999', '30,000+'],
    include_lowest=True,
)
volume_signal = (
    volume_rows.groupby('volume_bucket', observed=True)
    .agg(
        n=('content_id', 'size'),
        median_impressions=('impressions_90d', 'median'),
        median_clicks=('clicks_90d', 'median'),
        pages_with_click_pct=('clicks_90d', lambda values: 100 * (values > 0).mean()),
    )
    .reset_index()
)

print('Signal A — CTR versus position | MIXED')
display(position_signal.style.format({'median_ctr_pct': '{:.3f}', 'weighted_ctr_pct': '{:.3f}'}))
print('Signal B — impression volume | CONFIRMED')
display(volume_signal.style.format({
    'median_impressions': '{:,.0f}',
    'median_clicks': '{:,.0f}',
    'pages_with_click_pct': '{:.1f}',
}))

Signal A — CTR versus position | MIXED


,position_bucket,n,median_ctr_pct,weighted_ctr_pct
0,top_3,480,0.200,0.489
1,page_1,7084,0.240,0.348
2,striking,4459,0.170,0.351
3,page_3_5,4314,0.090,0.154
4,deep,389,0.000,0.037


Signal B — impression volume | CONFIRMED


,volume_bucket,n,median_impressions,median_clicks,pages_with_click_pct
0,1-499,13274,53,0,19.6
1,"500-2,999",8443,"1,230",2,72.2
2,"3,000-29,999",7205,"7,249",16,97.5
3,"30,000+",1078,"48,675",116,99.9


## 2. Build the ranked queue

**Rule in plain words:** review a page when it has at least 500 impressions, is already visible in positions 4–20, and its measured CTR is below the weighted CTR of pages in the same coarse position bucket. Rank it by estimated missed clicks: `impressions × positive CTR gap / 100`. The thresholds and weights are fixed by hand; nothing is fitted.

- One reason code: `visible_low_ctr`
- One action label: `review_title_and_meta`

The score estimates review upside under a peer-rate assumption. It does not predict causal uplift. The CSV is regenerated on every run and stays out of git by design.

In [2]:
score_inputs = ['impressions_90d', 'ctr', 'avg_position']
eligible = df.loc[
    (df['impressions_90d'] >= 500) & df['avg_position'].between(4, 20)
].copy()
eligible['position_bucket'] = pd.cut(
    eligible['avg_position'], bins=[3, 10, 20], labels=['page_1', 'striking']
)

benchmarks = (
    eligible.groupby('position_bucket', observed=True)
    .agg(impressions=('impressions_90d', 'sum'), clicks=('clicks_90d', 'sum'))
)
benchmarks['benchmark_ctr_pct'] = 100 * benchmarks['clicks'] / benchmarks['impressions']
eligible['benchmark_ctr_pct'] = (
    eligible['position_bucket'].map(benchmarks['benchmark_ctr_pct']).astype(float)
)
eligible['ctr_gap_pp'] = (eligible['benchmark_ctr_pct'] - eligible['ctr']).clip(lower=0)
eligible['baseline_score'] = eligible['impressions_90d'] * eligible['ctr_gap_pp'] / 100
eligible['severe_low_ctr_audit'] = (
    eligible['ctr'] <= 0.5 * eligible['benchmark_ctr_pct']
).astype(int)

queue = (
    eligible.loc[eligible['baseline_score'] > 0]
    .sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False])
    .reset_index(drop=True)
)
queue.insert(0, 'baseline_rank', np.arange(1, len(queue) + 1))
queue['reason_code'] = 'visible_low_ctr'
queue['action_label'] = 'review_title_and_meta'

queue_columns = [
    'baseline_rank', 'content_id', 'baseline_score', 'reason_code', 'action_label',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'position_bucket',
    'benchmark_ctr_pct', 'ctr_gap_pp', 'days_since_last_update', 'content_age_days',
    'content_type', 'main_intent',
]
queue_path = output_dir / 'baseline_action_score.csv'
queue[queue_columns].to_csv(queue_path, index=False)

base_rate = eligible['severe_low_ctr_audit'].mean()
precision_at_10 = queue.head(10)['severe_low_ctr_audit'].mean()
metrics = {
    'dataset_rows': int(len(df)),
    'position_signal_verdict': 'MIXED',
    'volume_signal_verdict': 'CONFIRMED',
    'eligible_rows': int(len(eligible)),
    'queue_rows': int(len(queue)),
    'severe_low_ctr_base_rate': round(float(base_rate), 4),
    'descriptive_precision_at_10': round(float(precision_at_10), 4),
    'reason_code': 'visible_low_ctr',
    'action_label': 'review_title_and_meta',
    'random_seed': RANDOM_SEED,
}
metrics_path = output_dir / 'ml07_baseline_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')

assert queue['reason_code'].nunique() == 1
assert queue['action_label'].nunique() == 1
assert queue['baseline_score'].is_monotonic_decreasing
print(f'Eligible pages: {len(eligible):,}; ranked review queue: {len(queue):,}.')
print(f'Descriptive severe-low-CTR base rate: {base_rate:.1%}.')
print(f'Descriptive precision@10: {precision_at_10:.1%} (not a future-outcome metric).')
print('Wrote work/outputs/baseline_action_score.csv and ml07_baseline_metrics.json.')
display(queue[queue_columns[:12]].head(10).style.format({
    'baseline_score': '{:,.1f}', 'ctr': '{:.2f}', 'avg_position': '{:.1f}',
    'benchmark_ctr_pct': '{:.3f}', 'ctr_gap_pp': '{:.3f}',
}))

Eligible pages: 10,939; ranked review queue: 7,542.
Descriptive severe-low-CTR base rate: 42.0%.
Descriptive precision@10: 90.0% (not a future-outcome metric).
Wrote work/outputs/baseline_action_score.csv and ml07_baseline_metrics.json.


,baseline_rank,content_id,baseline_score,reason_code,action_label,impressions_90d,clicks_90d,ctr,avg_position,position_bucket,benchmark_ctr_pct,ctr_gap_pp
0,1,content_5fe46e04994d,956.6,visible_low_ctr,review_title_and_meta,517715,741,0.14,4.2,page_1,0.325,0.185
1,2,content_36ff89c8214e,810.8,visible_low_ctr,review_title_and_meta,295097,154,0.05,7.3,page_1,0.325,0.275
2,3,content_c8e9d6ab9013,677.7,visible_low_ctr,review_title_and_meta,208678,0,0.00,9.7,page_1,0.325,0.325
3,4,content_c84a0ab98e90,658.1,visible_low_ctr,review_title_and_meta,223271,70,0.03,7.8,page_1,0.325,0.295
4,5,content_cb112fce36be,510.6,visible_low_ctr,review_title_and_meta,309910,492,0.16,5.6,page_1,0.325,0.165
5,6,content_73c54f78c06a,480.9,visible_low_ctr,review_title_and_meta,213963,211,0.10,4.7,page_1,0.325,0.225
6,7,content_453722754fea,440.9,visible_low_ctr,review_title_and_meta,140079,16,0.01,7.6,page_1,0.325,0.315
7,8,content_a7427266c305,431.9,visible_low_ctr,review_title_and_meta,201111,219,0.11,5.7,page_1,0.325,0.215
8,9,content_91652435f57a,422.5,visible_low_ctr,review_title_and_meta,159590,100,0.06,7.8,page_1,0.325,0.265
9,10,content_db5989a78dd3,396.1,visible_low_ctr,review_title_and_meta,345111,733,0.21,5.4,page_1,0.325,0.115


## 3. Top-10 skeptical review

Each row below has the required action, why it ranked, and a concrete failure condition. Confidence means confidence that the page deserves **human review**, not confidence that an edit will increase clicks. Pseudonymous content IDs are shown so the generated queue and review remain traceable without publishing client names or URLs.

In [3]:
wrong_if = [
    'The impressions come from broad or branded queries whose intent this page should not target.',
    'The position average hides a query mix dominated by SERP features or low-intent impressions.',
    'The zero-click extreme is a tracking, canonicalization, or page-to-query attribution problem.',
    'The update 20 days ago has not had enough time to be recrawled and measured.',
    'Transactional SERPs are crowded by ads or rich results, making the peer CTR unattainable.',
    'The recent update is still settling, so another snippet change would confound measurement.',
    'Sixteen clicks from 140k impressions reflects a reporting or URL-mapping issue, not copy quality.',
    'The target-keyword metadata is mismatched and the broad query mix makes the benchmark unfair.',
    'Commercial-intent SERP features depress attainable organic CTR below the bucket benchmark.',
    'The page was recently updated and its modest CTR gap disappears under a narrower peer group.',
]
top10 = queue.head(10).copy()
top10['confidence_note'] = np.where(
    (top10['ctr'] <= 0.10) & (top10['days_since_last_update'] >= 90),
    'high-priority audit',
    'medium; verify context',
)
top10['why_it_is_here'] = top10.apply(
    lambda row: (
        f"{row['impressions_90d']:,.0f} impressions at position {row['avg_position']:.1f}; "
        f"CTR {row['ctr']:.2f}% is {row['ctr_gap_pp']:.3f}pp below its peer benchmark."
    ),
    axis=1,
)
top10['what_would_make_it_wrong'] = wrong_if
review_columns = [
    'baseline_rank', 'content_id', 'action_label', 'reason_code', 'confidence_note',
    'why_it_is_here', 'what_would_make_it_wrong',
]
assert len(top10) == 10
assert top10[review_columns].notna().all().all()
pd.set_option('display.max_colwidth', 120)
display(top10[review_columns])

,baseline_rank,content_id,action_label,reason_code,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,content_5fe46e04994d,review_title_and_meta,visible_low_ctr,medium; verify context,"517,715 impressions at position 4.2; CTR 0.14% is 0.185pp below its peer benchmark.",The impressions come from broad or branded queries whose intent this page should not target.
1,2,content_36ff89c8214e,review_title_and_meta,visible_low_ctr,high-priority audit,"295,097 impressions at position 7.3; CTR 0.05% is 0.275pp below its peer benchmark.",The position average hides a query mix dominated by SERP features or low-intent impressions.
2,3,content_c8e9d6ab9013,review_title_and_meta,visible_low_ctr,high-priority audit,"208,678 impressions at position 9.7; CTR 0.00% is 0.325pp below its peer benchmark.","The zero-click extreme is a tracking, canonicalization, or page-to-query attribution problem."
3,4,content_c84a0ab98e90,review_title_and_meta,visible_low_ctr,medium; verify context,"223,271 impressions at position 7.8; CTR 0.03% is 0.295pp below its peer benchmark.",The update 20 days ago has not had enough time to be recrawled and measured.
4,5,content_cb112fce36be,review_title_and_meta,visible_low_ctr,medium; verify context,"309,910 impressions at position 5.6; CTR 0.16% is 0.165pp below its peer benchmark.","Transactional SERPs are crowded by ads or rich results, making the peer CTR unattainable."
5,6,content_73c54f78c06a,review_title_and_meta,visible_low_ctr,medium; verify context,"213,963 impressions at position 4.7; CTR 0.10% is 0.225pp below its peer benchmark.","The recent update is still settling, so another snippet change would confound measurement."
6,7,content_453722754fea,review_title_and_meta,visible_low_ctr,medium; verify context,"140,079 impressions at position 7.6; CTR 0.01% is 0.315pp below its peer benchmark.","Sixteen clicks from 140k impressions reflects a reporting or URL-mapping issue, not copy quality."
7,8,content_a7427266c305,review_title_and_meta,visible_low_ctr,medium; verify context,"201,111 impressions at position 5.7; CTR 0.11% is 0.215pp below its peer benchmark.",The target-keyword metadata is mismatched and the broad query mix makes the benchmark unfair.
8,9,content_91652435f57a,review_title_and_meta,visible_low_ctr,high-priority audit,"159,590 impressions at position 7.8; CTR 0.06% is 0.265pp below its peer benchmark.",Commercial-intent SERP features depress attainable organic CTR below the bucket benchmark.
9,10,content_db5989a78dd3,review_title_and_meta,visible_low_ctr,medium; verify context,"345,111 impressions at position 5.4; CTR 0.21% is 0.115pp below its peer benchmark.",The page was recently updated and its modest CTR gap disappears under a narrower peer group.


## 4. Weak picks and leakage check

The weakest top-ten picks are ranks 4, 6, 7, and 10 because they were updated only 20 days ago; a wait-and-measure decision may be better than another edit. Rank 10 also has a smaller CTR gap, so its position-bucket benchmark is easier to overturn with a narrower query or intent peer group. Rank 3 is high priority but high uncertainty: zero clicks from more than 200k impressions is large enough to demand a tracking and canonicalization check before anyone changes copy.

The score uses only `impressions_90d`, `ctr`, and `avg_position`. It does **not** use product flags, IDs, `trend_pct`, `trend_direction`, the derived decline label, or any last/previous-30-day trend inputs. The severe-low-CTR audit indicator is used only to describe concentration at rank 10; it is not an input to the score and is not a future validation label.

In [4]:
forbidden_inputs = {
    'content_id', 'client_id', 'trend_pct', 'trend_direction', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'severe_low_ctr_audit',
}
assert set(score_inputs).isdisjoint(forbidden_inputs)
assert score_inputs == ['impressions_90d', 'ctr', 'avg_position']
assert not any('flag' in column.lower() for column in score_inputs)

weak_pick_review = top10.loc[
    top10['baseline_rank'].isin([3, 4, 6, 7, 10]),
    ['baseline_rank', 'content_id', 'days_since_last_update', 'ctr_gap_pp',
     'what_would_make_it_wrong'],
]
print('Leakage check passed: three transparent contemporaneous score inputs; no flags, IDs, or trend/label fields.')
display(weak_pick_review.style.format({'ctr_gap_pp': '{:.3f}'}))

Leakage check passed: three transparent contemporaneous score inputs; no flags, IDs, or trend/label fields.


,baseline_rank,content_id,days_since_last_update,ctr_gap_pp,what_would_make_it_wrong
2,3,content_c8e9d6ab9013,104,0.325,"The zero-click extreme is a tracking, canonicalization, or page-to-query attribution problem."
3,4,content_c84a0ab98e90,20,0.295,The update 20 days ago has not had enough time to be recrawled and measured.
5,6,content_73c54f78c06a,20,0.225,"The recent update is still settling, so another snippet change would confound measurement."
6,7,content_453722754fea,20,0.315,"Sixteen clicks from 140k impressions reflects a reporting or URL-mapping issue, not copy quality."
9,10,content_db5989a78dd3,20,0.115,The page was recently updated and its modest CTR gap disappears under a narrower peer group.


## Self-check

- [x] Two signal checks have visible bucket tables and `n`; CTR-vs-position is flag-linked
- [x] Each signal has one allowed verdict: MIXED and CONFIRMED
- [x] One fixed rule has a score, one reason code, and one action label
- [x] The notebook writes `work/outputs/baseline_action_score.csv` on every run
- [x] Exactly ten ranked rows are reviewed with action, reason, confidence, and a failure condition
- [x] Weak picks are named and no future-window, label-derived, ID, or product-flag input is used
- [x] Claims remain observational, directional, and decision-support only
- [x] The notebook runs top to bottom without errors and publishes no client names, URLs, or private queries